## Changing Directory

In [1]:
import os
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-\\research'

In [2]:
os.chdir("..")

In [3]:
%pwd

'c:\\Users\\Aswin\\MLprojects\\Quality_Of_wine(EndtoEnd ML)\\Quality_Of_Wine-EndtoEnd-'

## config entity

In [ ]:
# config entity (DataTransformationConfig)

"""
An Entity is a simple Python class (usually a dataclass) that defines the structure of the data needed for this stage.

"""

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

## Configuration Manager

In [ ]:
# packages needed for configuration manager
from Quality_of_Wine.constants import *
from Quality_of_Wine.utils.common import read_yaml, create_directories

In [ ]:
# Configuration Manager

"""
The Configuration Manager is the "middleman" that reads your config.yaml and converts it into the Entity above.

Function: get_data_transformation_config()

How it works: It uses read_yaml (from your utils/common.py) to find the folder paths and returns the
DataTransformationConfig object populated with those real values.

"""
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    """
    This is often the entry point inside your component file.   

    Job: It initializes the DataTransformation class and triggers the processing logic.
    
    Relationship: It connects the Configuration Manager (which provides the paths) to the Component Class (which does the work).
    
    """
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
        )

        return data_transformation_config

## Data transfomation-Components

In [7]:
import os
from Quality_of_Wine import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [ ]:
# data Transformation

"""
This is the "Worker" or the Component. It contains the actual math and logic.

Job: It takes the raw data (e.g., train.csv) and performs operations like scaling, encoding, 
or handling missing values.

Internal Methods: It usually has a method like initiate_data_transformation() which reads the 
data and returns the transformed arrays (train/test sets).

"""

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config


    def train_test_spliting(self):
        data = pd.read_csv(self.config.data_path)

        train, test = train_test_split(data)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"),index = False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"),index = False)

        logger.info("Splited data into training and test sets")
        logger.info(train.shape)
        logger.info(test.shape)

        print(train.shape," is the shape of the training data")
        print(test.shape," is the shape of the testing data")
        

## Data Transformation Pipeline

In [ ]:
#pipeline(for data transformation)

"""
The Pipeline is the "conveyor belt" that coordinates this specific stage.

The Workflow:
~Initialize the Configuration Manager.
~Get the Data Transformation Config (The Entity).
~Call the Data Transformation Component.
~Save the transformed data as an Artifact (e.g., train.npy) so the Model Trainer can use it later.
"""

try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_spliting()
except Exception as e:
    raise e

[2026-03-04 22:02:09,984: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-04 22:02:10,000: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-04 22:02:10,007: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-04 22:02:10,020: INFO: common: created directory at: artifacts]


[2026-03-04 22:02:10,022: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-04 22:02:10,171: INFO: 2261784035: Splited data into training and test sets]
[2026-03-04 22:02:10,171: INFO: 2261784035: (1199, 12)]
[2026-03-04 22:02:10,176: INFO: 2261784035: (400, 12)]
(1199, 12)
(400, 12)


## modular coding for data transformation in src folder

In [ ]:
"""
~Succcessfully ran the DATA TRANSFORMATION part of the project in reseach folder.

~End To End MLOPS Data Science Project Implementation With Deployment(YT Channel)=https://youtu.be/pxk1Fr33-L4?si=YXioCQhqeFOWQ_az

~Now it has to be covered into modular coding.
   the process and where it has to copied is mentioned below

   *Step 01: entity/config_entity.py
            ~ In the entity/config_entity.py copy and paste the code below the entity of data validation
              code from config entity markdown above, just the class.
   
   *Step 02: src/config/configuration.py
            ~ In the src/config/configuration.py copy and paste the def function of the code from Configuration Manager markdown
              above, the packages are already in the file written during data ingestion phase.
            ~ the other package must be imported from src/entity.

   *Step 03: create a new file in src/components/data_transformation.py(new file)
            ~ In the src/components/data_transformation.py copy and paste the code from Data transformation markdown above with packages
             and import needed packages.
   
   *Step 04: create a new file in src/pipelines/stage_03_data_transformation.py(new file)
            ~ Before copying the pipeline code, some code have to coded first that is already in src/pipelines/stage_03_data_transformation.ipynb
              and a logic has to be included where only if the status of the data validation is True then the pipeline must run.
   
   *Step 05: To test the pipeline from the above step the execution must be done from main.py
            ~ From the code from the src/pipelines/stage_02_data_validation.ipynb copy and paste the code with the packages in the main.py.
            ~ Delete the artifacts folder.
            ~ And execute it in the terminal -python main.py.

"""